# A3 ABO · 01 Chuẩn bị dữ liệu và cache

Thiên duyệt cặp ảnh–tiêu đề và nhóm sản phẩm. Khoa chốt quy tắc che từ bằng train. Notebook tạo manifest và hai cache dùng chung.

Colab dùng A3_ABO_team.zip trong MyDrive/A3_ABO. Jupyter local chạy từ checkout A3 hoặc đặt biến A3_REPO. Dùng cùng A3_WORK, tên cohort và A3_RUN_ID cho các notebook.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Edit paths here if using another Drive folder or a checkout outside the current directory.
WORK = Path(os.environ.get("A3_WORK", "/content/drive/MyDrive/A3_ABO" if ON_COLAB else "runs/a3/team")).resolve()
REPO = Path(os.environ.get("A3_REPO", ".")).resolve()
if not (REPO / "src/dlbench/a3/curation.py").is_file():
    if not ON_COLAB and (REPO.parent / "src/dlbench/a3/curation.py").is_file():
        REPO = REPO.parent
    elif ON_COLAB:
        bundle = WORK / "A3_ABO_team.zip"
        if bundle.is_file():
            digest = hashlib.sha256(bundle.read_bytes()).hexdigest()[:16]
            REPO = Path("/content") / f"a3-source-{digest}"
            if not (REPO / "src/dlbench/a3/curation.py").is_file():
                shutil.unpack_archive(bundle, REPO)
        elif (WORK / "setup.py").is_file() and (WORK / "src/dlbench/a3/team_batches.py").is_file():
            REPO = WORK
        else:
            raise FileNotFoundError(f"Đặt A3_ABO_team.zip trong WORK hoặc chỉnh WORK. Đang dùng: {WORK}")
    else:
        raise FileNotFoundError("Set A3_REPO to the A3 checkout or extracted source ZIP.")

if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[a3]"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)
env = {**os.environ, "PYTHONPATH": source + os.pathsep + os.environ.get("PYTHONPATH", "")}

def run_a3(*args, capture_error=False):
    command = [sys.executable, "-X", "utf8", "-m", "dlbench.a3.cli", *map(str, args)]
    if not capture_error:
        subprocess.run(command, cwd=REPO, env=env, check=True)
        return
    result = subprocess.run(command, cwd=REPO, env=env, capture_output=True,
                            text=True, encoding="utf-8", errors="replace")
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode:
        details = result.stderr.strip()
        errors = [line for line in details.splitlines() if ": error:" in line]
        raise RuntimeError(errors[-1] if errors else details or f"Lệnh thất bại (mã {result.returncode}).")
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr)

WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"
IMAGE_ROOT = DATA / "images"
FROZEN = WORK / "cohorts" / "abo-v1"
IMAGE_CACHE = FROZEN / "image.npz"
TITLE_CACHE = FROZEN / "title.npz"
MANIFEST = FROZEN / "manifest.csv"
RUN_ID = os.environ.get("A3_RUN_ID", "abo-v1-main")
SUITE = WORK / "runs" / RUN_ID
INPUT_ARGS = ("--manifest", MANIFEST, "--image-cache", IMAGE_CACHE, "--title-cache", TITLE_CACHE)
print(f"Source: {REPO}\nWorkspace: {WORK}")

def require_cohort(*paths):
    missing = [path.name for path in paths if not path.is_file()]
    if not missing:
        return
    decisions = WORK / "reviews" / "abo-v1" / "decisions.csv"
    if "freeze.json" in missing or "manifest.csv" in missing:
        if FROZEN.exists():
            action = "Thư mục cohort đang thiếu file. Chép lại nguyên cohort đã chốt hoặc chỉnh WORK."
        elif decisions.is_file():
            action = (f"Audit đã có tại {decisions}. Duyệt và lưu file, rồi điền REVIEWER và "
                      "chạy ô chốt manifest trong Notebook 01 Prepare.")
        else:
            action = "Chạy Notebook 01 Prepare để chốt dữ liệu, hoặc chỉnh WORK tới workspace đã chuẩn bị."
    else:
        action = "Chạy ô tạo cache trong Notebook 01 Prepare với ENCODE = True."
    raise RuntimeError(f"Thiếu {', '.join(missing)} ở {FROZEN}. {action}")

## Tạo danh sách cần review

Notebook tải metadata và ảnh nhỏ cần dùng từ ABO nếu chưa có. Dữ liệu được quy thuộc cho Amazon.com và bộ dữ liệu ABO của Collins và cộng sự ([nguồn và CC BY 4.0](https://amazon-berkeley-objects.s3.us-east-1.amazonaws.com/index.html)).

In [ ]:
from dlbench.a3.data import read_csv

AUDIT = WORK / "reviews" / "abo-v1"
if not (AUDIT / "audit.json").exists():
    run_a3("prepare-data", "--work", DATA)
    run_a3("audit-data", "--candidates", DATA / "candidates.csv", "--image-root",
           IMAGE_ROOT, "--output", AUDIT)
print((AUDIT / "audit.json").read_text(encoding="utf-8"))
print(AUDIT / "decisions.csv")

## Duyệt ảnh và nhóm sản phẩm

Mở **decisions.csv** bằng trình sửa CSV có thể giữ nguyên ID. Với một ID, dùng **keep** hoặc **drop** và ghi lý do khi loại. Với hai ID, dùng **merge** để gom, **separate** để buộc tách nhóm, hoặc **keep** để giữ cả hai mà không thêm liên kết hay ràng buộc tách. Có thể thêm dòng **manual** để loại thêm mẫu hoặc ghép các biến thể tìm thấy ngoài danh sách. Giữ nguyên các cột ID và reason của dòng có sẵn. Với xung đột nhãn ở ảnh trùng pixel, chỉ giữ nhãn có thể xác nhận hoặc loại cả nhóm.

Ô dưới xem từng trang. Đổi **OFFSET** hoặc **REASON_CONTAINS** để chuyển giữa mẫu ngẫu nhiên theo lớp và cặp nghi gần trùng/biến thể.

In [ ]:
from collections import Counter
from IPython.display import display
from PIL import Image, ImageOps

OFFSET, LIMIT = 0, 5
REASON_CONTAINS = "stratified_sample"   # near_image, brand_model, title_stem, exact_label_conflict, hoặc ""
inventory = {row["sample_id"]: row for row in read_csv(AUDIT / "inventory.csv")}
all_cases = read_csv(AUDIT / "decisions.csv")
pending = Counter(row["reason"] for row in all_cases if not row["decision"].strip())
print(f"Còn {sum(pending.values()):,}/{len(all_cases):,} dòng chưa có quyết định.")
if pending:
    print("Theo lý do:", dict(pending.most_common()))
cases = [row for row in all_cases if REASON_CONTAINS in row["reason"]]
print(f"{len(cases)} dòng phù hợp. Tệp quyết định: {AUDIT / 'decisions.csv'}")
for case in cases[OFFSET:OFFSET + LIMIT]:
    print(case["reason"], case["decision"] or "pending")
    for sample in (case["sample_id"], case["related_id"]):
        if sample:
            row = inventory[sample]
            print(sample, row["label"], row.get("brand"), row.get("model_number"), row["title"])
            with Image.open(IMAGE_ROOT / row["image_path"]) as source_image:
                thumb = ImageOps.exif_transpose(source_image).convert("RGB")
                thumb.thumbnail((180, 180))
                display(thumb)

## Chốt manifest và tạo cache

Điền **REVIEWER** sau khi lưu quyết định. Xem thống kê từ manifest đã chốt. Nếu thêm biến thể tên lớp, lưu một cụm từ trên mỗi dòng trong **extra_mask_phrases.txt** tại thư mục in ra. Chọn các cụm này từ title train. Tệp có thể bỏ trống khi dùng danh sách tên lớp mặc định. Nếu đổi review hoặc title policy sau khi lưu cache, đổi phiên bản cohort và run.

In [ ]:
from dlbench.a3.curation import verified_freeze

REVIEWER = ""
if FROZEN.exists():
    require_cohort(FROZEN / "freeze.json", MANIFEST)
elif REVIEWER.strip():
    from dlbench.a3.data import read_csv

    decisions = AUDIT / "decisions.csv"
    cases = read_csv(decisions)
    pending = [row for row in cases if not row["decision"].strip()]
    if pending:
        first = pending[0]
        pair = " / ".join(value for value in (first["sample_id"], first["related_id"]) if value)
        raise ValueError(
            f"Còn {len(pending):,}/{len(cases):,} dòng chưa điền decision trong {decisions}. "
            f"Dòng đầu: {pair} ({first['reason']}). REVIEWER là tên người duyệt. "
            "Lưu quyết định vào đúng file CSV rồi chạy lại ô này."
        )
    run_a3("freeze-data", "--audit-dir", AUDIT, "--image-root", IMAGE_ROOT,
           "--output", FROZEN, "--reviewer", REVIEWER, capture_error=True)
rows = verified_freeze(FROZEN) if MANIFEST.is_file() else None
if rows is None:
    print(f"Duyệt {AUDIT / 'decisions.csv'}, điền REVIEWER rồi chạy lại ô này.")
else:
    print((FROZEN / "freeze.json").read_text(encoding="utf-8"))
    print("Tệp cụm từ mở rộng:", FROZEN / "extra_mask_phrases.txt")
    print("Ví dụ title thuộc train:")
    for row in [row for row in rows if row["split"] == "train"][:15]:
        print(row["label"], row["title"])

In [ ]:
from dlbench.a3.cli import _device
from dlbench.a3.features import encode_images, encode_titles, inspect_title_policy, load_cache, save_cache

ENCODER_DEVICE = "auto"
ENCODE = False   # Đặt True khi đã chốt chính sách che title.
if ENCODE:
    require_cohort(FROZEN / "freeze.json", MANIFEST)
    rows = verified_freeze(FROZEN)
    receipt = json.loads((FROZEN / "freeze.json").read_text(encoding="utf-8"))
    phrases_path = FROZEN / "extra_mask_phrases.txt"
    phrases = phrases_path.read_text(encoding="utf-8-sig").splitlines() if phrases_path.exists() else ()
    policy = inspect_title_policy(rows, phrases)
    device = _device(ENCODER_DEVICE)
    if not IMAGE_CACHE.is_file():
        verified_freeze(FROZEN, image_root=IMAGE_ROOT)
        vectors, metadata = encode_images(rows, IMAGE_ROOT, 32, device)
        save_cache(IMAGE_CACHE, rows, "image", vectors,
                   {**metadata, "image_files_sha256": receipt["image_files_sha256"]})
    if not TITLE_CACHE.is_file():
        vectors, metadata = encode_titles(rows, 32, device, phrases=phrases)
        save_cache(TITLE_CACHE, rows, "title", vectors, metadata)
    if load_cache(IMAGE_CACHE, rows, "image")[1]["encoder"].get("image_files_sha256") != receipt["image_files_sha256"]:
        raise ValueError("Cache ảnh khác inventory đã duyệt.")
    if load_cache(TITLE_CACHE, rows, "title")[1]["encoder"].get("mask_policy") != policy:
        raise ValueError("Cache title khác chính sách che từ đã chọn.")
    print("Cache đã kiểm tra:", IMAGE_CACHE, TITLE_CACHE)
    print(json.dumps(policy, indent=2, ensure_ascii=False))
else:
    print("Sau khi chốt manifest, đặt ENCODE = True và chạy lại ô này.")